# BirdCLEF 2026 — Submission Notebook


In [1]:
# ============================================================
# Cell 1 — Imports & Configuration
# ============================================================
import os
import pickle
import warnings
import glob
from pathlib import Path
from typing import Optional
from dataclasses import dataclass

import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio.transforms as T

import timm

warnings.filterwarnings('ignore')


@dataclass
class Config:
    # Paths
    data_dir: Path = Path('/kaggle/input/competitions/birdclef-2026')
    models_dir: Path = Path('/kaggle/input/datasets/katie090902/cnntransformer-cnn')  # your uploaded dataset
    output_dir: Path = Path('/kaggle/working')

    # Audio parameters — must match training
    sample_rate: int = 32000
    segment_duration: float = 5.0
    hop_length: int = 320
    n_fft: int = 2048
    n_mels: int = 128
    fmin: int = 20
    fmax: int = 16000

    # Model parameters — must match training
    model_name: str = 'tf_efficientnet_b0_ns'
    num_classes: int = 234
    batch_size: int = 16  # lower than training since CPU only

    # Device — automatically CPU since GPU is off
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'

    @property
    def segment_samples(self) -> int:
        return int(self.segment_duration * self.sample_rate)


config = Config()
print(f'Device: {config.device}')
print(f'Models dir: {config.models_dir}')
print(f'Data dir:   {config.data_dir}')

# Verify model weights exist
for fname in ['cnn_best.pth', 'hybrid_best.pth', 'label_encoder.pkl']:
    path = config.models_dir / fname
    exists = path.exists()
    print(f'  {"✓" if exists else "✗"} {fname}')
    if not exists:
        raise FileNotFoundError(f'{fname} not found at {path}. Add your trained-models dataset via + Add Data.')

Device: cpu
Models dir: /kaggle/input/datasets/katie090902/cnntransformer-cnn
Data dir:   /kaggle/input/competitions/birdclef-2026
  ✓ cnn_best.pth
  ✓ hybrid_best.pth
  ✓ label_encoder.pkl


In [2]:
# ============================================================
# Cell 2 — Model Definitions (pretrained=False, no download)
# ============================================================

class AudioProcessor:
    def __init__(self, config: Config):
        self.config = config
        self.mel_transform = T.MelSpectrogram(
            sample_rate=config.sample_rate,
            n_fft=config.n_fft,
            hop_length=config.hop_length,
            n_mels=config.n_mels,
            f_min=config.fmin,
            f_max=config.fmax,
            power=2.0,
        )
        self.db_transform = T.AmplitudeToDB(stype='power', top_db=80)

    def process_file(self, filepath, offset: float = 0.0, duration: Optional[float] = None) -> torch.Tensor:
        duration = duration or self.config.segment_duration
        try:
            audio, sr = librosa.load(
                filepath, sr=self.config.sample_rate,
                offset=offset, duration=duration, mono=True
            )
        except Exception as e:
            print(f'Error loading {filepath}: {e}')
            audio = np.zeros(int(self.config.sample_rate * duration), dtype=np.float32)

        # Normalize
        max_val = np.abs(audio).max()
        if max_val > 0:
            audio = audio / max_val

        # Pad or truncate
        target = int(self.config.sample_rate * duration)
        if len(audio) > target:
            audio = audio[:target]
        elif len(audio) < target:
            audio = np.pad(audio, (0, target - len(audio)))

        waveform = torch.from_numpy(audio).float().unsqueeze(0)
        mel_spec = self.mel_transform(waveform)
        mel_spec_db = self.db_transform(mel_spec)
        mel_spec_db = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min() + 1e-6)
        return mel_spec_db.squeeze(0)


class SoundscapeDataset(Dataset):
    def __init__(self, filepaths, config):
        self.config = config
        self.audio_processor = AudioProcessor(config)
        self.segments = []
        for fp in filepaths:
            for start_sec in range(0, 60, 5):
                self.segments.append({
                    'filepath': fp,
                    'start_sec': start_sec,
                    'row_id': f'{fp.stem}_{start_sec + 5}'
                })

    def __len__(self):
        return len(self.segments)

    def __getitem__(self, idx):
        seg = self.segments[idx]
        mel = self.audio_processor.process_file(seg['filepath'], offset=seg['start_sec'], duration=5.0)
        return mel.unsqueeze(0).repeat(3, 1, 1), seg['row_id']


class AttentionPooling(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(in_features, in_features // 4), nn.ReLU(),
            nn.Linear(in_features // 4, 1),
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        weights = F.softmax(self.attention(x), dim=1)
        return (x * weights).sum(dim=1)


class BirdCLEFCNN(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        # pretrained=False — no internet download needed
        self.backbone = timm.create_model(
            config.model_name, pretrained=False, in_chans=3, num_classes=0, global_pool=''
        )
        with torch.no_grad():
            dummy = torch.randn(1, 3, config.n_mels, 500)
            features = self.backbone(dummy)
            self.feature_dim = features.shape[1]
        self.attention_pool = AttentionPooling(self.feature_dim)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.feature_dim, config.num_classes),
        )

    def forward(self, x):
        features = self.backbone(x)
        features = features.mean(dim=2)
        pooled = self.attention_pool(features)
        return self.classifier(pooled)


class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads=8, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(int(dim * mlp_ratio), dim), nn.Dropout(dropout),
        )

    def forward(self, x):
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + attn_out
        return x + self.mlp(self.norm2(x))


class HybridModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        # pretrained=False — no internet download needed
        self.cnn = timm.create_model(
            'tf_efficientnet_b0_ns', pretrained=False, in_chans=1, num_classes=0, global_pool=''
        )
        with torch.no_grad():
            dummy = torch.randn(1, 1, config.n_mels, 500)
            cnn_out = self.cnn(dummy)
            self.cnn_channels = cnn_out.shape[1]
            self.cnn_h = cnn_out.shape[2]
            self.cnn_w = cnn_out.shape[3]
        embed_dim = 256
        self.proj = nn.Conv2d(self.cnn_channels, embed_dim, kernel_size=1)
        num_positions = self.cnn_h * self.cnn_w
        self.pos_embed = nn.Parameter(torch.zeros(1, num_positions, embed_dim))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.transformer = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads=4, mlp_ratio=4.0, dropout=0.1) for _ in range(4)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(embed_dim, config.num_classes),
        )

    def forward(self, x):
        x = x[:, 0:1, :, :]
        batch_size = x.shape[0]
        features = self.cnn(x)
        features = self.proj(features)
        features = features.flatten(2).transpose(1, 2)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        features = torch.cat([cls_tokens, features], dim=1)
        pos_embed = torch.cat([
            torch.zeros(1, 1, features.shape[-1], device=features.device),
            self.pos_embed
        ], dim=1)
        features = features + pos_embed
        for block in self.transformer:
            features = block(features)
        features = self.norm(features)
        return self.head(features[:, 0])


print('Model classes defined.')

Model classes defined.


In [3]:
# ============================================================
# Cell 3 — Load Label Encoder & Model Weights
# ============================================================

# Load label encoder
with open(config.models_dir / '/kaggle/input/datasets/katie090902/cnntransformer-cnn/label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)
print(f'Label encoder loaded: {len(label_encoder.classes_)} species')

# Instantiate models
cnn_model    = BirdCLEFCNN(config)
hybrid_model = HybridModel(config)

# Load saved weights
cnn_ckpt    = torch.load(config.models_dir / '/kaggle/input/datasets/katie090902/cnntransformer-cnn/cnn_best.pth',    map_location=config.device)
hybrid_ckpt = torch.load(config.models_dir / '/kaggle/input/datasets/katie090902/cnntransformer-cnn/hybrid_best.pth', map_location=config.device)

cnn_model.load_state_dict(cnn_ckpt['model_state_dict'])
hybrid_model.load_state_dict(hybrid_ckpt['model_state_dict'])

cnn_model    = cnn_model.to(config.device).eval()
hybrid_model = hybrid_model.to(config.device).eval()

print(f'CNN loaded    — best epoch: {cnn_ckpt["epoch"]+1}, val_loss: {cnn_ckpt["val_loss"]:.4f}')
print(f'Hybrid loaded — best epoch: {hybrid_ckpt["epoch"]+1}, val_loss: {hybrid_ckpt["val_loss"]:.4f}')

# Use the model with lower val_loss as best, ensemble both
print('\nBoth models loaded and ready.')

Label encoder loaded: 234 species
CNN loaded    — best epoch: 3, val_loss: 0.0018
Hybrid loaded — best epoch: 3, val_loss: 0.0023

Both models loaded and ready.


In [4]:
# ============================================================
# Cell 4 — Run Inference & Save Submission
# ============================================================

# Load sample submission for row_ids and column order
sample_sub  = pd.read_csv(config.data_dir / 'sample_submission.csv')
species_cols = [c for c in sample_sub.columns if c != 'row_id']
print(f'Sample submission: {len(sample_sub)} rows, {len(species_cols)} species')

# Find test soundscape files
test_dir   = config.data_dir / 'test_soundscapes'
test_files = sorted(test_dir.glob('*.ogg'))
print(f'Test soundscapes found: {len(test_files)}')

if len(test_files) == 0:
    # Fallback — happens during local dev when test files are placeholder only
    print('WARNING: No test files found. Saving uniform-probability fallback submission.')
    submission_df = sample_sub.copy()
    for col in species_cols:
        submission_df[col] = 1 / len(species_cols)
else:
    # Build dataset and loader
    test_dataset = SoundscapeDataset(test_files, config)
    test_loader  = DataLoader(test_dataset, batch_size=config.batch_size,
                              shuffle=False, num_workers=2, pin_memory=False)
    print(f'Total segments to predict: {len(test_dataset)}')

    all_row_ids = []
    all_probs   = []

    with torch.no_grad():
        for inputs, row_ids in tqdm(test_loader, desc='Inference'):
            inputs = inputs.to(config.device)

            # Ensemble: average CNN + Hybrid predictions
            probs_cnn    = torch.sigmoid(cnn_model(inputs))
            probs_hybrid = torch.sigmoid(hybrid_model(inputs))
            probs        = ((probs_cnn + probs_hybrid) / 2).cpu().numpy()

            all_row_ids.extend(row_ids)
            all_probs.append(probs)

    all_probs = np.vstack(all_probs)

    # Build submission dataframe
    pred_df = pd.DataFrame({'row_id': all_row_ids})
    for i, species in enumerate(label_encoder.classes_):
        pred_df[species] = all_probs[:, i]

    # Align to sample_submission row order and columns
    submission_df = sample_sub[['row_id']].merge(pred_df, on='row_id', how='left')
    submission_df = submission_df.reindex(columns=['row_id'] + species_cols)
    submission_df[species_cols] = submission_df[species_cols].fillna(1 / len(species_cols))

# Save
out_path = config.output_dir / 'submission.csv'
submission_df.to_csv(out_path, index=False)
print(f'\nsubmission.csv saved! Shape: {submission_df.shape}')
print(submission_df.head(3))

Sample submission: 3 rows, 234 species
Test soundscapes found: 0

submission.csv saved! Shape: (3, 235)
                                    row_id   1161364    116570   1176823  \
0   BC2026_Test_0001_S05_20250227_010002_5  0.004274  0.004274  0.004274   
1  BC2026_Test_0001_S05_20250227_010002_10  0.004274  0.004274  0.004274   
2  BC2026_Test_0001_S05_20250227_010002_15  0.004274  0.004274  0.004274   

    1491113   1595929    209233     22930     22956     22961  ...   whnjay1  \
0  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  ...  0.004274   
1  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  ...  0.004274   
2  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  ...  0.004274   

     whtdov   whwpic1    y00678    yebcar   yebela1    yecmac    yecpar  \
0  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274   
1  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274   
2  0.004274  0.004274  0.004274  0.004274  0.